# M10 — Make Metrics Reflect Consequences

**Begin with the decision:** a positive prediction triggers a manual asset inspection. A false positive consumes scarce capacity; a false negative misses an imminent failure. Only after orienting those consequences will we define metrics.

Whole-first loop: **consequences → confusion outcomes → metrics → threshold sweep → decision utility → locked policy → test evidence**.


In [ ]:
from pathlib import Path
import csv
import platform
import sys

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'missions.json').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the LearningOS-AI repository')

ROOT = find_repo_root()
DATASET = ROOT / 'datasets' / 'M10' / 'asset_alert_scores.csv'
CONSEQUENCES = ROOT / 'datasets' / 'M10' / 'consequence_matrix.csv'
print({'python': sys.version.split()[0], 'platform': platform.system(), 'cpu_only': True, 'network_calls': 0})


## 1. Consequences before metric definitions

**Predict before running:** Which error should be more expensive? Write the operational meaning of TP, FP, TN and FN, and predict whether the chosen policy should favor catching failures or avoiding inspections.

The fixture uses teaching cost units, not money. Correct outcomes have zero *marginal* cost only inside this bounded exercise. Real decisions may need action cost, delay, heterogeneous harm, subgroup constraints and uncertainty.


In [ ]:
def load_consequence_costs(path):
    with path.open(encoding='utf-8', newline='') as handle:
        rows = list(csv.DictReader(handle))
    costs = {row['outcome']: float(row['cost_units']) for row in rows}
    assert set(costs) == {'TP', 'FP', 'TN', 'FN'}
    return rows, costs

consequence_rows, COSTS = load_consequence_costs(CONSEQUENCES)
CAPACITY_LIMIT = 0.50
for row in consequence_rows:
    print(row['outcome'], '=>', row['operational_meaning'], '| cost:', row['cost_units'])
print('maximum alert rate:', CAPACITY_LIMIT)


## 2. Fixed scores, separate validation and test evidence

The classifier is already scored. M10 changes its decision layer, not its learned parameters. Validation data may select a threshold. Test outcomes remain untouched until the threshold is locked. The score is a ranking signal here; no calibration claim is made.


In [ ]:
def load_scores(path):
    with path.open(encoding='utf-8', newline='') as handle:
        rows = list(csv.DictReader(handle))
    return [
        {'case_id': row['case_id'], 'split': row['split'], 'score': float(row['risk_score']), 'actual': int(row['failure_within_30d'])}
        for row in rows
    ]

all_rows = load_scores(DATASET)
validation = [row for row in all_rows if row['split'] == 'validation']
test = [row for row in all_rows if row['split'] == 'test']
assert len(validation) == len(test) == 30
assert sum(row['actual'] for row in validation) == sum(row['actual'] for row in test) == 6
print({'validation_rows': len(validation), 'validation_positives': sum(row['actual'] for row in validation), 'prevalence': sum(row['actual'] for row in validation) / len(validation)})


## 3. Confusion matrix at one operating threshold

A score at or above the threshold triggers the positive action.

**Predict before running:** At threshold 0.50, manually count TP, FP, TN and FN. Then predict whether precision and recall will be equal.


In [ ]:
def confusion_counts(rows, threshold):
    counts = {'TP': 0, 'FP': 0, 'TN': 0, 'FN': 0}
    for row in rows:
        predicted = int(row['score'] >= threshold)
        actual = row['actual']
        outcome = 'TP' if predicted and actual else 'FP' if predicted else 'FN' if actual else 'TN'
        counts[outcome] += 1
    assert sum(counts.values()) == len(rows)
    return counts

def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else 0.0

def metrics_from_counts(counts, costs):
    tp, fp, tn, fn = (counts[key] for key in ('TP', 'FP', 'TN', 'FN'))
    total = tp + fp + tn + fn
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    specificity = safe_divide(tn, tn + fp)
    f1 = safe_divide(2 * precision * recall, precision + recall)
    expected_cost = sum(counts[key] * costs[key] for key in counts)
    no_alert_cost = (tp + fn) * costs['FN']
    return {
        **counts,
        'accuracy': safe_divide(tp + tn, total),
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'f1': f1,
        'false_positive_rate': 1.0 - specificity,
        'alert_rate': safe_divide(tp + fp, total),
        'expected_cost': expected_cost,
        'cost_per_case': safe_divide(expected_cost, total),
        'value_vs_no_alert': no_alert_cost - expected_cost,
    }


In [ ]:
at_half = metrics_from_counts(confusion_counts(validation, 0.50), COSTS)
print(at_half)
assert {key: at_half[key] for key in ('TP', 'FP', 'TN', 'FN')} == {'TP': 3, 'FP': 3, 'TN': 21, 'FN': 3}


### Interpret, do not merely name

- **Accuracy** = `(TP + TN) / all`: useful only when equal-error weighting matches the decision.
- **Precision** = `TP / (TP + FP)`: the hit rate among actions taken.
- **Recall / sensitivity** = `TP / (TP + FN)`: coverage of actual positive events.
- **Specificity** = `TN / (TN + FP)`: coverage of actual negative events.
- **F1** = `2PR / (P + R)` = `2TP / (2TP + FP + FN)`: a harmonic mean that penalizes a low component. F1 omits TN and treats precision and recall symmetrically; those are modeling choices, not universal utility.

This notebook uses 0.0 for an undefined rate. A production report should also flag the undefined denominator rather than silently treating it as evidence of bad or good performance.


## 4. Imbalance can make inaction look attractive

**Predict before running:** With 20% positives, what accuracy does an always-negative policy achieve? What are its recall and expected cost?


In [ ]:
all_negative = metrics_from_counts(confusion_counts(validation, 1.01), COSTS)
print(all_negative)
assert all_negative['accuracy'] == 0.80
assert all_negative['recall'] == 0.0
assert all_negative['expected_cost'] == 108.0


## 5. One score ranking, many action policies

**Predict before running:** As the threshold falls, which counts and rates can only increase, which can only decrease, and which may move non-monotonically?


In [ ]:
def threshold_table(rows, costs):
    thresholds = sorted({row['score'] for row in rows} | {0.0, 1.01}, reverse=True)
    table = []
    for threshold in thresholds:
        metrics = metrics_from_counts(confusion_counts(rows, threshold), costs)
        table.append({'threshold': threshold, **metrics})
    return table

def print_threshold_rows(rows):
    keys = ('threshold', 'TP', 'FP', 'TN', 'FN', 'accuracy', 'precision', 'recall', 'specificity', 'f1', 'alert_rate', 'expected_cost')
    print(' '.join(f'{key:>13}' for key in keys))
    for row in rows:
        values = [f"{row[key]:13.3f}" if isinstance(row[key], float) else f"{row[key]:13}" for key in keys]
        print(' '.join(values))

validation_table = threshold_table(validation, COSTS)
display_thresholds = {1.01, 0.92, 0.72, 0.54, 0.50, 0.38, 0.31, 0.18, 0.0}
print_threshold_rows([row for row in validation_table if row['threshold'] in display_thresholds])


## 6. Controlled failure — optimize an attractive metric

**Predict before running:** Will maximum accuracy choose the same threshold as minimum expected cost? Predict the dominant error and the direction of utility harm.

The deliberately faulty policy maximizes validation accuracy and hides its confusion counts. The repaired policy minimizes stated cost among thresholds satisfying the 50% alert-capacity limit.


In [ ]:
accuracy_choice = max(validation_table, key=lambda row: (row['accuracy'], row['threshold']))
feasible = [row for row in validation_table if row['alert_rate'] <= CAPACITY_LIMIT]
utility_choice = min(feasible, key=lambda row: (row['expected_cost'], -row['recall'], row['threshold']))
print('accuracy-selected:', accuracy_choice)
print('consequence-selected:', utility_choice)
assert accuracy_choice['accuracy'] > utility_choice['accuracy']
assert accuracy_choice['expected_cost'] > utility_choice['expected_cost']
assert accuracy_choice['threshold'] == 0.92
assert utility_choice['threshold'] == 0.18
assert accuracy_choice['expected_cost'] == 90.0
assert utility_choice['expected_cost'] == 18.0
print('root cause: accuracy weights each row equally; the decision contract does not')


## 7. F1 is useful, but it is still a choice

**Predict before running:** Because F1 omits TN and symmetrically balances precision with recall, will its optimum equal the cost optimum under a 9:1 FN:FP ratio?


In [ ]:
f1_choice = max(validation_table, key=lambda row: (row['f1'], row['threshold']))
print('F1-selected:', f1_choice)
print('consequence-selected:', utility_choice)
assert f1_choice['threshold'] == 0.31
assert f1_choice['expected_cost'] > utility_choice['expected_cost']
assert f1_choice['f1'] > utility_choice['f1']


## 8. ROC and precision-recall concepts

Every threshold produces a ROC point `(false-positive rate, recall)` and a PR point `(recall, precision)`. ROC AUC measures how often a random positive outranks a random negative. Average precision summarizes precision as recall increases. Neither selects an operating threshold, encodes cost, or enforces capacity.

**Predict before running:** With rare positives, which view exposes alert purity directly? Will changing only the action threshold alter ranking AUC?


In [ ]:
def roc_auc_pairwise(rows):
    positives = [row['score'] for row in rows if row['actual'] == 1]
    negatives = [row['score'] for row in rows if row['actual'] == 0]
    wins = sum(1.0 if positive > negative else 0.5 if positive == negative else 0.0 for positive in positives for negative in negatives)
    return safe_divide(wins, len(positives) * len(negatives))

def average_precision(rows):
    ranked = sorted(rows, key=lambda row: (-row['score'], row['case_id']))
    positives = sum(row['actual'] for row in ranked)
    true_positives = 0
    precision_at_positive = []
    for rank, row in enumerate(ranked, start=1):
        if row['actual']:
            true_positives += 1
            precision_at_positive.append(true_positives / rank)
    return safe_divide(sum(precision_at_positive), positives)

roc_auc = roc_auc_pairwise(validation)
avg_precision = average_precision(validation)
curve_rows = [row for row in validation_table if row['threshold'] in {1.01, 0.92, 0.72, 0.54, 0.31, 0.18, 0.0}]
print('ROC points:', [(row['false_positive_rate'], row['recall']) for row in curve_rows])
print('PR points:', [(row['recall'], row['precision']) for row in curve_rows])
print({'ROC_AUC': round(roc_auc, 3), 'average_precision': round(avg_precision, 3), 'positive_prevalence': 0.20})
assert 0.0 <= roc_auc <= 1.0 and 0.0 <= avg_precision <= 1.0


## 9. Lock on validation, then inspect test once

**Predict before running:** Will the locked threshold reproduce validation cost on the test split? State why a difference must be reported rather than repaired by retuning on test.


In [ ]:
LOCKED_THRESHOLD = utility_choice['threshold']
test_result = metrics_from_counts(confusion_counts(test, LOCKED_THRESHOLD), COSTS)
print({'locked_threshold': LOCKED_THRESHOLD, 'validation': utility_choice, 'test': test_result})
assert LOCKED_THRESHOLD == 0.18
assert {key: test_result[key] for key in ('TP', 'FP', 'TN', 'FN')} == {'TP': 5, 'FP': 9, 'TN': 15, 'FN': 1}
print('The threshold remains locked; this test difference is evidence, not a retuning instruction.')


## 10. Sensitivity and revisit conditions

Expected cost is only as defensible as its assumptions.

**Predict before running:** As FN cost rises while FP cost and evidence stay fixed, should the selected threshold generally rise or fall? Where can the capacity constraint interrupt that pattern?


In [ ]:
sensitivity = []
for fn_cost in (4.0, 8.0, 18.0, 40.0):
    scenario_costs = {**COSTS, 'FN': fn_cost}
    table = threshold_table(validation, scenario_costs)
    feasible_rows = [row for row in table if row['alert_rate'] <= CAPACITY_LIMIT]
    choice = min(feasible_rows, key=lambda row: (row['expected_cost'], -row['recall'], row['threshold']))
    sensitivity.append({'fn_cost': fn_cost, 'threshold': choice['threshold'], 'recall': choice['recall'], 'alert_rate': choice['alert_rate'], 'expected_cost': choice['expected_cost']})
for row in sensitivity:
    print(row)
assert [row['fn_cost'] for row in sensitivity] == [4.0, 8.0, 18.0, 40.0]


## 11. Turn analysis into a governed decision

The notebook is evidence, not approval. Complete `missions/M10/adr_prompt.md` to record the consequential metric/threshold decision, alternatives, assumptions, owner, monitoring, rollback and quantitative revisit conditions. Then use `missions/M10/review_brief.md` for formal engineering review; disposition every comment **Accept / Reject / Defer** with written reasoning.

Finally, close this notebook and complete the **No-AI Gate** in `missions/M10/no_ai_gate.md`. It presents unseen food-allergen consequences, capacity and threshold evidence. Choose and defend a metric/threshold independently; no answer is prefilled. Complete `missions/M10/assessment.yaml` for transfer.
